# Libaries

In [1]:
!pip install -qU langchain
!pip install -q "markitdown[all]"
!pip install -q langchain-huggingface sentence_transformers
!pip install -q langchain-chroma
!pip install -qU langchain-text-splitters
!pip install -q langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-adk 2.7.1 requires opentelemet

# Indexing


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)


# Embedding

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-mpnet-base-v2"
)



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# File System

In [4]:
from langchain_chroma import Chroma
from markitdown import MarkItDown
import os
from langchain_core.documents import Document

def process_file(file):
    global vector_store

    if file is None:
        return "Please upload a document first."


    md = MarkItDown()
    marked = md.convert(file)


    doc = Document(
        page_content=marked.text_content,
        metadata={"source": os.path.basename(file)}
    )


    all_splitt = splitter.split_documents([doc])
    vector_store = Chroma(
    embedding_function = embedding,
    persist_directory='./chroma-db',
    )
    vector_store.add_documents(all_splitt)
    return ""




# Retriever


In [8]:
def retreive(query:str,k:int = 2):
  source_docs = vector_store.similarity_search(query,k=k)
  doc_content = ""
  for i in source_docs:
    doc_content += f'Source: {i.metadata}'
    doc_content += f'Content: {i.page_content}'
    doc_content += '\n'
  return source_docs,doc_content


# LLM

In [12]:
from langchain.chat_models import init_chat_model
from google.colab import userdata
import gradio as gr
system_prompt = """
You are DocuMind AI, an intelligent document-based question-answering assistant.

Your task is to answer the user's questions using the provided context retrieved from the document database.

Instructions:
1. Answer the user's question using only the information available in the provided context.
2. Do not make up information or use external knowledge.
3. If the answer is not available in the context, clearly say:
   "I couldn't find the answer in the provided documents."
4. Provide clear, accurate, and easy-to-understand answers.
5. If relevant, organize your answer using bullet points or numbered lists.
6. Do not repeat unnecessary information.
7. Use the source metadata to identify which document the information came from.
8. Always provide answers in plain text.
9. Do not use Markdown formatting, headings, bold text, asterisks, or Markdown bullet syntax.
10. Use simple sentences and paragraphs.
11.give me context about that query according to document not only 3 or 4 lines give me the 10 lines.

Retrieved Context:
{docs_content}
"""
api_key = userdata.get('API')
model = init_chat_model(
    'google_genai:gemini-3.5-flash',
    api_key = api_key
)
def DocuMindAi(user_query:str):
  source_docs,docs_content = retreive(user_query)
  formated_prompt = system_prompt.format(docs_content = docs_content)
  messages = [
      {'role':'system','content':formated_prompt},
      {'role':'user','content':user_query}
  ]
  sources = [
        {
            "document": doc.metadata.get("source", "Unknown"),
            "content": doc.page_content
        }
        for doc in source_docs
    ]
  response = model.invoke(messages)
  return response.text,sources

def run_command(file, question):

    if file is None:
        gr.Warning("Please upload a document first!")
        return "", {}

    try:
        file = process_file(file)
        gr.Info("Document uploaded successfully!")

    except Exception as e:
        gr.Error(f"Document upload failed: {e}")
        return "", {}

    try:
        answer, sources = DocuMindAi(question)
        return answer, sources

    except Exception as e:
        gr.Error(f"Answer generation failed: {e}")
        return "", {}

# UI

In [13]:
import gradio as gr
demo = gr.Interface(
    fn = run_command,
    inputs = [
        gr.File(label='Upload Document',file_count='single',type='filepath'),
        gr.Textbox(
            label='Ask Your Question',
            placeholder="Ask anything about your document..."
        )
    ],
    outputs = [
        gr.Textbox(label='Content'),
        gr.JSON(label='Source')
    ],
    title = 'DocuMind AI',
    description = 'Upload your document and ask questions about its content.'
)
demo.launch(debug=True, show_error=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://10c0758941509a0360.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://10c0758941509a0360.gradio.live
